In [ ]:
# Mount Google Drive to access input/output data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# ============================================================
# Rename Shifted ECOSTRESS GeoTIFFs with Local Time Suffix
# ============================================================
# ECOSTRESS filenames encode acquisition time as UTC using the
# format 'doyYYYYDDDHHMMSS' (year + day-of-year + time).
#
# This cell:
#   1. Parses the UTC timestamp embedded in each 'Shifted_*.tif' filename.
#   2. Converts it to local time by applying a UTC offset.
#   3. Renames the file by stripping the 'Shifted_' prefix and
#      appending the local datetime as 'YYYYMMDD_HHMM'.
#
# Update UTC_OFFSET_HOURS to match your study area's UTC offset.
# Example offsets:
#   Belize / Central Standard Time : -6
#   US Pacific Standard Time       : -8
#   US Eastern Standard Time       : -5
# ============================================================

import os
import re
from datetime import datetime, timedelta
from glob import glob

# --- User configuration ---
# Set this to the folder containing your Shifted_*.tif files.
input_dir = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST"

# UTC offset for your study area (in hours). Negative = west of UTC.
UTC_OFFSET_HOURS = -6


def extract_local_time(filename, utc_offset_hours):
    """
    Parse the UTC acquisition timestamp from an ECOSTRESS filename and
    convert it to local time.

    ECOSTRESS filenames contain a timestamp segment in the format:
        doyYYYYDDDHHMMSS
    where YYYY = year, DDD = day-of-year, HH = hour, MM = minute, SS = second.

    Parameters
    ----------
    filename        : str   - ECOSTRESS GeoTIFF filename (basename only).
    utc_offset_hours: int   - Hours to add to UTC to get local time (e.g., -6 for UTC-6).

    Returns
    -------
    str - Local datetime string formatted as 'YYYYMMDD_HHMM'.

    Raises
    ------
    ValueError - If the expected timestamp pattern is not found in the filename.
    """
    match = re.search(r'doy(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})', filename)
    if match:
        year, doy, hour, minute, second = match.groups()
        utc_dt = datetime.strptime(f"{year}{doy}{hour}{minute}{second}", "%Y%j%H%M%S")
        local_dt = utc_dt + timedelta(hours=utc_offset_hours)
        return local_dt.strftime("%Y%m%d_%H%M")
    else:
        raise ValueError(f"Could not extract datetime from: {filename}")


# --- Rename all Shifted_*.tif files ---
tif_files = glob(os.path.join(input_dir, "Shifted_*.tif"))

for filepath in tif_files:
    filename = os.path.basename(filepath)

    # Convert UTC timestamp in filename to local time string
    local_time_str = extract_local_time(filename, UTC_OFFSET_HOURS)

    # Strip the 'Shifted_' prefix added during geolocation correction
    base_name = filename.replace("Shifted_", "")
    name_no_ext, ext = os.path.splitext(base_name)

    # Append local datetime to the filename before the extension
    new_filename = f"{name_no_ext}_{local_time_str}{ext}"
    new_path = os.path.join(input_dir, new_filename)

    os.rename(filepath, new_path)
    print(f"Renamed: {filename}  →  {new_filename}")


Renamed to: ECO_L2T_LSTE.002_LST_doy2023294085443_aid0001_16N_20231021_0254.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2024074231352_aid0001_16N_20240314_1713.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2024116071158_aid0001_16N_20240425_0111.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023130021040_aid0001_16N_20230509_2010.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023138084646_aid0001_16N_20230518_0246.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023145194125_aid0001_16N_20230525_1341.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023167112748_aid0001_16N_20230616_0527.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023167112656_aid0001_16N_20230616_0526.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023217151531_aid0001_16N_20230805_0915.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023267192031_aid0001_16N_20230924_1320.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023259223038_aid0001_16N_20230916_1630.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023278152238_aid0001_16N_20231005_0922.tif
Renamed to: ECO_L2T_LSTE.002_LST_doy2023

In [ ]:
# ============================================================
# Build a Timestamp Inventory CSV for Renamed ECOSTRESS Files
# ============================================================
# After renaming, this cell scans all *.tif files in the folder,
# extracts UTC and local acquisition times from each filename,
# classifies each scene as Day or Night based on local hour,
# and saves a summary CSV sorted chronologically by UTC time.
#
# Day/Night classification boundary:
#   Day   : local hour >= 6 and < 18
#   Night : all other hours
#
# Update UTC_OFFSET_HOURS below to match your study area.
# ============================================================

import os
import re
import pandas as pd
from datetime import datetime, timedelta
from glob import glob

# --- User configuration ---
input_dir = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST"

# UTC offset for your study area (in hours). Negative = west of UTC.
UTC_OFFSET_HOURS = -6


def extract_datetimes(filename, utc_offset_hours):
    """
    Parse the UTC acquisition timestamp from an ECOSTRESS filename and
    return both the UTC and local datetime objects.

    Parameters
    ----------
    filename         : str - ECOSTRESS GeoTIFF filename (basename only).
    utc_offset_hours : int - Hours to add to UTC to get local time.

    Returns
    -------
    utc_time   : datetime or None - UTC acquisition time.
    local_time : datetime or None - Local acquisition time.
    Returns (None, None) if the timestamp pattern is not found.
    """
    match = re.search(r'doy(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})', filename)
    if match:
        year, doy, hour, minute, second = match.groups()
        utc_time   = datetime.strptime(f"{year}{doy}{hour}{minute}{second}", "%Y%j%H%M%S")
        local_time = utc_time + timedelta(hours=utc_offset_hours)
        return utc_time, local_time
    return None, None


def classify_day_or_night(dt):
    """
    Classify an acquisition time as 'Day' or 'Night' based on local hour.

    Parameters
    ----------
    dt : datetime - Local acquisition datetime.

    Returns
    -------
    str - 'Day' if 06:00–17:59 local time, 'Night' otherwise.
    """
    return "Day" if 6 <= dt.hour < 18 else "Night"


# --- Scan files and extract timestamps ---
records  = []
tif_files = glob(os.path.join(input_dir, "*.tif"))

for filepath in tif_files:
    filename = os.path.basename(filepath)
    utc_dt, local_dt = extract_datetimes(filename, UTC_OFFSET_HOURS)

    if utc_dt:  # Skip files where timestamp extraction failed
        records.append({
            "Filename"         : filename,
            "UTC Time"         : utc_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "Local Time (24h)" : local_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "Local Time (12h)" : local_dt.strftime("%Y-%m-%d %I:%M %p"),
            "Day/Night"        : classify_day_or_night(local_dt)
        })

# --- Build DataFrame, sort chronologically, and save ---
df = pd.DataFrame(records)
df.sort_values(by="UTC Time", inplace=True)

output_csv = os.path.join(input_dir, "file_timestamps.csv")
df.to_csv(output_csv, index=False)
print(f"Timestamp inventory saved to: {output_csv}")

# Preview the first few rows
df.head()


,Filename,UTC Time,Belize Time (24h),Belize Time (12h),Day/Night
3,ECO_L2T_LSTE.002_LST_doy2023130021040_aid0001_...,2023-05-10 02:10:40,2023-05-09 20:10:40,2023-05-09 08:10 PM,Night
25,ECO_L2T_LSTE.002_LST_doy2023134003242_aid0001_...,2023-05-14 00:32:42,2023-05-13 18:32:42,2023-05-13 06:32 PM,Night
38,ECO_L2T_LSTE.002_LST_doy2023137225523_aid0001_...,2023-05-17 22:55:23,2023-05-17 16:55:23,2023-05-17 04:55 PM,Day
4,ECO_L2T_LSTE.002_LST_doy2023138084646_aid0001_...,2023-05-18 08:46:46,2023-05-18 02:46:46,2023-05-18 02:46 AM,Night
46,ECO_L2T_LSTE.002_LST_doy2023141075835_aid0001_...,2023-05-21 07:58:35,2023-05-21 01:58:35,2023-05-21 01:58 AM,Night
